In [10]:
from dotenv import load_dotenv
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents



In [14]:
load_dotenv()
openai_client = OpenAI()

# read the data from the github repo
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# files = reader.read()
# documents = []

# for file in files:
#     doc = file.parse()
#     documents.append(doc)

documents = [file.parse() for file in reader.read()]

# chunks = chunk_documents(documents)

KeyboardInterrupt: 

In [22]:
from embedder import Embedder

embedder = Embedder(path="../models/Xenova/all-MiniLM-L6-v2")

simple_q_embedding = embedder.encode("How does approximate nearest neighbor search work?")
simple_q_embedding[0]

np.float64(-0.020582036807885073)

In [56]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [ ]:
page = "02-vector-search/lessons/07-sqlitesearch-vector.md"

content = next((doc['content'] for doc in documents if doc['filename'] == page), None)
# indexed = {document['filename']: document for document in documents}
# indexed.get(page)

content_embedding = embedder.encode(content)
print(content_embedding.dot(simple_q_embedding))

0.361070280302606


In [ ]:
# q3
chunks = chunk_documents(documents, size=2000, step=1000)
chunk_documents_embeddings = embedder.encode_batch([chunk['content'] for chunk in chunks])


In [44]:
print(chunk_documents_embeddings)

[[-0.08756473  0.01836385 -0.08122418 ...  0.03053823 -0.02172771
   0.03277497]
 [ 0.02436193 -0.10619476  0.03307318 ...  0.01430081 -0.00125541
   0.04325696]
 [-0.01780485  0.03103095  0.00856106 ...  0.0222022  -0.03375529
   0.04288229]
 ...
 [ 0.00980343  0.04912254  0.01207489 ... -0.09453995 -0.06321278
   0.04775798]
 [-0.03622024  0.06821856 -0.01540897 ... -0.00271628  0.01875559
   0.01007469]
 [-0.02975658 -0.00552574 -0.03531848 ...  0.01044231  0.02297966
  -0.01966068]]


In [53]:
scores = chunk_documents_embeddings.dot(simple_q_embedding)
chunks[scores.argmax()]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [68]:
#q4 minsearch/VectorSearch
from minsearch import VectorSearch

vector_search = VectorSearch()
vector_search.fit(chunk_documents_embeddings, chunks)

def vecSearch(query, embedder: Embedder, vecSearch: VectorSearch, num_results=1):
  query_embedding = embedder.encode(query)
  return vecSearch.search(query_embedding, num_results=num_results)

q4 = embedder.encode("What metric do we use to evaluate a search engine?")
found_q4 = vector_search.search(q4, num_results=1)
print(found_q4)
print(vecSearch("What metric do we use to evaluate a search engine?", embedder, vector_search))


[{'start': 0, 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [58]:
vector_search2 = VectorSearch()
content_embeds = embedder.encode_batch([doc['content'] for doc in documents])
vector_search2.fit(content_embeds, documents)

q4 = embedder.encode("What metric do we use to evaluate a search engine?")
vector_search2.search(q4, num_results=1)

[{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query

In [ ]:
#q5
from minsearch import Index

index = Index(
  text_fields=["content"]
)
index.fit(chunks)


In [81]:

q5 = "How do I store vectors in PostgreSQL?"

def textSearch(query, index: Index, num_results=1):
  return index.search(query, num_results=num_results)
tsq5_results = textSearch(q5, index, num_results=5)
tsq5 = [doc['filename'] for doc in tsq5_results]

vsq5_results = vecSearch(q5, embedder, vector_search, num_results=5)
vsq5 = [doc['filename'] for doc in vsq5_results]


set(vsq5).difference(set(tsq5))




{'02-vector-search/lessons/08-pgvector.md'}

In [84]:
#q6 hybrid search
# Reciprocal Rank Fusion (RRF) - position of each document in each list
# RRF(d) = sum over lists of  1 / (k + rank(d))
# 
# 'sum over lists' means we go through each list and, for each list where the document is found, we add 1 / (k + rank(d)) to the score
def rrf(result_lists, k=60, num_results=5):
  scores = {}
  docs = {}

  for results in result_lists:
    for rank, doc in enumerate(results):
      key = (doc['filename'], doc['start'])
      scores[key] = scores.get(key, 0) + 1 / (k + rank)
      docs[key] = doc

  ranked = sorted(scores, key=scores.get, reverse=True)
  return [docs[key] for key in ranked[:num_results]]

q6 = "How do I give the model access to tools?"
tsq6_results = textSearch(q6, index, num_results=5)
vsq6_results = vecSearch(q6, embedder, vector_search, num_results=5)

results = rrf([tsq6_results, vsq6_results])
print(results[0])



{'start': 4000, 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function cal